In [2]:
'''
abbreviation notes: TDE (tidal disruption event), TC (TDE catalog), PSC (post-starburst catalog),
    SMBHB (supermassive black hole binary)
'''

'\nabbreviation notes: TDE (tidal disruption event), TC (TDE catalog), PSC (post-starburst catalog),\n    SMBHB (supermassive black hole binary)\n'

In [4]:
import os
import otter

from astropy.coordinates import SkyCoord
from astropy import units as u

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [5]:
db = otter.Otter()

Attempting to login to https://otter.idies.jhu.edu/api with the following credentials:
username: user-guest
password: test


In [6]:
'''
initial function
returns list of potential galaxy canidates
'''
# note: only concerned with TC for now

# initialize canidate list as objects classified as TDEs and within redshift 0.03
cands = db.query(classification="TDE", minz=0, maxz=0.03, class_confidence_threshold=1)

# initalize lists to be stored in dataframe
name_list = []
rs_list = []
cord_ra_list = []
cord_dec_list = []
hosts_info = []
confidence = []
bibcode = []

# iterate through all transients returned by initial query
i = 0
while i < len(cands):
    object = cands[i]
    
    name = object.default_name
    name_list.append(name)
    
    rs = float(object.get_redshift())
    rs_list.append(rs)
    
    cords = object.get_skycoord()
    ra = cords.ra.deg
    cord_ra_list.append(ra)
    dec = cords.dec.deg
    cord_dec_list.append(dec)

    conf = object.get_classification()
    confidence.append(conf[1]) # filter for confidences > 1
    bibcode.append(conf[2])
    
    info = object.get_host() # does OTTER have an object of class host assosiated with the transient?
    # need to know this to know if OTTER can return photometric information
    if info == []:
        hosts_info.append(False)
    else:
        hosts_info.append(True)

    i += 1


# initialize data frame storing all possible canidates
cands_df = pd.DataFrame(
    {"Name": name_list, "redshift": rs_list, "confidence": confidence, "ra": cord_ra_list, "dec":cord_dec_list, 
     "Host info in OTTER?": hosts_info, "Bibcode": bibcode}
)

cands_df = cands_df.set_index("Name")

cands_df

,redshift,confidence,ra,dec,Host info in OTTER?,Bibcode
Name,,,,,,
NGC 5905,0.011240,3.1,228.847167,55.517108,True,"[1996A&A...309L..35B, 2017ApJ...838..149A, 202..."
ARP 299-B AT1,0.010411,3.3,172.129115,58.561329,False,[2018Sci...361..482M]
XMMSL1 J0740-85,0.017300,3.3,115.034125,-85.658681,True,"[2017A&A...598A..29S, 2017ApJ...835...64G, 201..."
XMMSL2 J144605.0+685735,0.029000,3.3,221.520833,68.958611,True,"[2019A&A...630A..98S, 2023PASP..135c4101G, ['2..."
2016fnl,0.016300,3.3,7.487667,32.893750,True,"[2016ATel.9433....1G, 2016TNSCR.615....1Y, 201..."
2022gri,0.028000,3.3,109.586643,33.994853,False,"[2022TNSAN..99....1Y, 2022TNSCR1156....1Y, 202..."
2022dsb,0.023000,3.3,235.590633,-22.670580,False,"[2022TNSCR.584....1F, 2024ApJ...974..241A, 202..."
2019azh,0.022000,3.3,123.320627,22.648301,True,"[2019TNSCR.287....1B, 2019TNSTR...2....1S, 202..."
2020wey,0.027410,3.3,136.357958,61.802538,False,"[2020TNSCR3228....1A, 2023ApJ...942....9H, 202..."


In [7]:
first_row = cands_df.iloc[0]

first_row

redshift                                                         0.01124
confidence                                                           3.1
ra                                                            228.847167
dec                                                            55.517108
Host info in OTTER?                                                 True
Bibcode                [1996A&A...309L..35B, 2017ApJ...838..149A, 202...
Name: NGC 5905, dtype: object

In [8]:
cands_df.to_csv("OtterCands.csv")